In [1]:
import pandas as pd
import os
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
DATASET_PATH = "../datasets/supplement_dataset"
MODEL_PATH = "../models/supplement"

user_profiles_path = os.path.join(DATASET_PATH, "user_profiles.csv")

df = pd.read_csv(user_profiles_path)

df.head()

,user_id,age,gender,height_cm,weight_kg,BMI,bmi,health_condition,medication,goal,recommended_category,budget_range,activity_level
0,U0001,58,Female,178.4,78.2,24.6,24.6,NaN,NaN,Heart Health,Omega-3,10000-25000,Low
1,U0002,45,Female,149.2,59.7,26.8,26.8,NaN,NaN,Immunity,Vitamin,0-5000,Low
2,U0003,30,Male,176.6,63.9,20.5,20.5,NaN,NaN,Weight Loss,Performance,5000-10000,Low
3,U0004,45,Female,152.0,61.7,26.7,26.7,Arthritis,Naproxen,Arm Muscle Growth,Protein,5000-10000,Moderate
4,U0005,23,Female,161.2,58.9,22.7,22.7,NaN,NaN,Immunity,Vitamin,0-5000,Moderate


In [3]:
print(df.columns.tolist())
print(df.shape)

['user_id', 'age', 'gender', 'height_cm', 'weight_kg', 'BMI', 'bmi', 'health_condition', 'medication', 'goal', 'recommended_category', 'budget_range', 'activity_level']
(3000, 13)


In [4]:
if "bmi" not in df.columns and "BMI" in df.columns:
    df["bmi"] = df["BMI"]

df["bmi"] = pd.to_numeric(df["bmi"], errors="coerce")
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["height_cm"] = pd.to_numeric(df["height_cm"], errors="coerce")
df["weight_kg"] = pd.to_numeric(df["weight_kg"], errors="coerce")

df = df.dropna(subset=["age", "height_cm", "weight_kg", "bmi", "recommended_category"])

df.shape

(3000, 13)

In [5]:
features = [
    "age",
    "gender",
    "height_cm",
    "weight_kg",
    "bmi",
    "health_condition",
    "medication",
    "goal",
    "budget_range",
    "activity_level"
]

target = "recommended_category"

X = df[features]
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)
print(y.value_counts())

X shape: (3000, 10)
y shape: (3000,)
recommended_category
General Wellness    781
Protein             518
Joint Support       471
Performance         264
Vitamin             248
Collagen Protein    245
BCAA                240
Omega-3             233
Name: count, dtype: int64


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

Training rows: 2400
Testing rows: 600


In [7]:
numeric_features = ["age", "height_cm", "weight_kg", "bmi"]
categorical_features = ["gender", "health_condition", "medication", "goal", "budget_range", "activity_level"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

In [8]:
pipeline.fit(X_train, y_train)

print("Model training completed!")

Model training completed!


In [9]:
y_pred = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 1.0

Classification Report:
                  precision    recall  f1-score   support

            BCAA       1.00      1.00      1.00        48
Collagen Protein       1.00      1.00      1.00        49
General Wellness       1.00      1.00      1.00       156
   Joint Support       1.00      1.00      1.00        94
         Omega-3       1.00      1.00      1.00        47
     Performance       1.00      1.00      1.00        53
         Protein       1.00      1.00      1.00       104
         Vitamin       1.00      1.00      1.00        49

        accuracy                           1.00       600
       macro avg       1.00      1.00      1.00       600
    weighted avg       1.00      1.00      1.00       600



In [10]:
sample_user = pd.DataFrame([{
    "age": 24,
    "gender": "Female",
    "height_cm": 160,
    "weight_kg": 55,
    "bmi": 21.5,
    "health_condition": "None",
    "medication": "None",
    "goal": "muscle gain",
    "budget_range": "10000-25000",
    "activity_level": "Moderate"
}])

predicted_category = pipeline.predict(sample_user)[0]

print("Predicted Supplement Category:", predicted_category)

Predicted Supplement Category: General Wellness


In [11]:
text_columns = [
    "gender",
    "health_condition",
    "medication",
    "goal",
    "budget_range",
    "activity_level",
    "recommended_category"
]

for col in text_columns:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.lower()

df["goal"].value_counts()

goal
arm muscle growth    284
energy boost         270
weight loss          264
endurance            262
general wellness     249
immunity             248
hair and skin        245
recovery             240
bone health          240
muscle gain          234
heart health         233
joint support        231
Name: count, dtype: int64

In [12]:
def map_goal_to_category(goal):
    goal = str(goal).lower().strip()

    if any(word in goal for word in ["muscle", "body", "bodybuilding", "strength", "arms", "bulk", "gain"]):
        return "protein"
    elif any(word in goal for word in ["recovery", "workout recovery"]):
        return "bcaa"
    elif any(word in goal for word in ["heart", "cholesterol", "cardio"]):
        return "omega-3"
    elif any(word in goal for word in ["immunity", "immune"]):
        return "vitamin"
    elif any(word in goal for word in ["joint", "bone"]):
        return "joint support"
    elif any(word in goal for word in ["digestive", "digestion", "gut"]):
        return "digestive"
    elif any(word in goal for word in ["skin", "hair", "beauty", "collagen"]):
        return "collagen protein"
    elif any(word in goal for word in ["fat", "weight loss", "slim", "cut"]):
        return "performance"
    elif any(word in goal for word in ["energy", "wellness", "normal"]):
        return "general wellness"
    else:
        return "general wellness"

df["recommended_category"] = df["goal"].apply(map_goal_to_category)

df["recommended_category"].value_counts()

recommended_category
general wellness    781
protein             518
joint support       471
performance         264
vitamin             248
collagen protein    245
bcaa                240
omega-3             233
Name: count, dtype: int64

In [13]:
sample_user = pd.DataFrame([{
    "age": 24,
    "gender": "female",
    "height_cm": 160,
    "weight_kg": 55,
    "bmi": 21.5,
    "health_condition": "none",
    "medication": "none",
    "goal": "muscle gain",
    "budget_range": "10000-25000",
    "activity_level": "moderate"
}])

predicted_category = pipeline.predict(sample_user)[0]

print("Predicted Supplement Category:", predicted_category)

Predicted Supplement Category: General Wellness


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Features and target
features = [
    "age",
    "gender",
    "height_cm",
    "weight_kg",
    "bmi",
    "health_condition",
    "medication",
    "goal",
    "budget_range",
    "activity_level"
]

target = "recommended_category"

X = df[features]
y = df[target]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Preprocessing
numeric_features = ["age", "height_cm", "weight_kg", "bmi"]
categorical_features = [
    "gender",
    "health_condition",
    "medication",
    "goal",
    "budget_range",
    "activity_level"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# Model
model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

# Train again
pipeline.fit(X_train, y_train)

# Evaluate
y_pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Test sample
sample_user = pd.DataFrame([{
    "age": 24,
    "gender": "female",
    "height_cm": 160,
    "weight_kg": 55,
    "bmi": 21.5,
    "health_condition": "none",
    "medication": "none",
    "goal": "muscle gain",
    "budget_range": "10000-25000",
    "activity_level": "moderate"
}])

predicted_category = pipeline.predict(sample_user)[0]

print("Predicted Supplement Category:", predicted_category)

Accuracy: 1.0
                  precision    recall  f1-score   support

            bcaa       1.00      1.00      1.00        48
collagen protein       1.00      1.00      1.00        49
general wellness       1.00      1.00      1.00       156
   joint support       1.00      1.00      1.00        94
         omega-3       1.00      1.00      1.00        47
     performance       1.00      1.00      1.00        53
         protein       1.00      1.00      1.00       104
         vitamin       1.00      1.00      1.00        49

        accuracy                           1.00       600
       macro avg       1.00      1.00      1.00       600
    weighted avg       1.00      1.00      1.00       600

Predicted Supplement Category: protein


In [16]:
import os
import pickle

MODEL_PATH = "../models/supplement"
os.makedirs(MODEL_PATH, exist_ok=True)

model_file = os.path.join(MODEL_PATH, "supplement_model.pkl")

with open(model_file, "wb") as f:
    pickle.dump(pipeline, f)

print("Model saved successfully:", model_file)

Model saved successfully: ../models/supplement\supplement_model.pkl
